In [54]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [55]:
#Dataset load
iris = datasets.load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

print("Input shape:", X.shape)
print("Labels shape:", y.shape)

Input shape: (150, 4)
Labels shape: (150, 1)


In [56]:
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

print(y[:5])

[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


In [57]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [58]:
#Feature Select
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [59]:
#Initialize Parameters

np.random.seed(42)

input_size = 4
hidden_size = 5
output_size = 3

W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

In [60]:
#Activation function

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [61]:
#Forword propagation

def forward(X):
    global Z1, A1, Z2, A2

    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    return A2

In [63]:
#Backward Propagation

def loss(y_true, y_pred):
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-9)) / m

In [65]:
#Backpropagation

def backward(X, y, lr=0.1):
    global W1, b1, W2, b2

    m = X.shape[0]

    dZ2 = A2 - y
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * A1 * (1 - A1)

    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

In [66]:
#gradient checking

def gradient_check(X, y, epsilon=1e-7):
    global W1

    original_value = W1[0,0]

    W1[0,0] = original_value + epsilon
    loss_plus = loss(y, forward(X))

    W1[0,0] = original_value - epsilon
    loss_minus = loss(y, forward(X))

    W1[0,0] = original_value

    approx_grad = (loss_plus - loss_minus) / (2 * epsilon)
    print("Numerical Gradient:", approx_grad)

In [67]:
# Training model
epochs = 1000

for i in range(epochs):
    y_pred = forward(X_train)
    l = loss(y_train, y_pred)
    backward(X_train, y_train)

    if i % 200 == 0:
        print(f"Epoch {i}, Loss: {l:.4f}")

Epoch 0, Loss: 1.0989
Epoch 200, Loss: 0.9018
Epoch 400, Loss: 0.4604
Epoch 600, Loss: 0.3561
Epoch 800, Loss: 0.2820


In [70]:
#Evaluation

y_test_pred = forward(X_test)

predictions = np.argmax(y_test_pred, axis=1)
actual = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == actual)

print("Accuracy:", accuracy)

Accuracy: 0.9666666666666667


In [71]:
gradient_check(X_train, y_train)

Numerical Gradient: -0.009150475654973178
